# 🏆 BLOCK 3 — DEEP LEARNING: INTRACRANIAL HEMORRHAGE (ICH) ASSESSMENT
### Pipeline Huấn Luyện & Đánh Giá Tối Tân: Tích Hợp Kỹ Thuật Quán Quân RSNA & MICCAI INSTANCE 2022
**Nhóm thực hiện:** Vũ Minh Hoàng & Nguyễn Đình Hữu Minh  
**Môi trường chạy:** Google Colab GPU (Tesla T4 16GB VRAM)

---

### 🌟 CÁC CẢI TIẾN ĐỘT PHÁ ĐÃ TÍCH HỢP TỪ CÁC ĐỘI TOP 1 THẾ GIỚI:
1. **Tiền xử lý 3 Cửa sổ CT (Multi-Window CT):**
   - Chuyển đổi CT Hounsfield Units (HU) thành ảnh 3 kênh RGB: **Brain Window** ($L=40, W=80$), **Subdural Window** ($L=80, W=200$), **Bone Window** ($L=600, W=2800$).
   - Giữ nguyên cấu trúc tổn thương mỏng (Anti-aliased thin lesion preservation) khi downsample.
2. **Bộ Phân Loại Chuỗi 3D (Quán quân RSNA 2019 - SeuTao):**
   - **ResNet-34 Feature Extractor** (trích 512 đặc trưng/lát cắt) kết hợp **Bidirectional GRU** (xâu chuỗi ngữ cảnh 3D xuyên suốt các lát cắt theo trục Z).
   - Loss: `BCEWithLogitsLoss` có trọng số bù trừ mất cân bằng các phân loại hiếm (EDH, SDH).
3. **Bộ Phân Đoạn Nâng Cao (Quán quân NVIDIA MICCAI INSTANCE 2022):**
   - **scSE Attention U-Net** (Spatial and Channel Squeeze & Excitation): Tự động triệt tiêu tín hiệu xương sọ và tập trung vào mô tổn thương.
   - **Focal Tversky Loss** ($\alpha=0.7, \beta=0.3, \gamma=1.33$): Phạt nặng lỗi bỏ sót các đốm máu nhỏ, dải máu mỏng (khắc phục triệt để lỗi bỏ sót SAH rãnh não), kiểm soát co gọn bờ tổn thương (tránh over-segmentation).
4. **Hậu xử lý Lọc Xương Vật Lý (Physical HU Gating & Noise Filtering):**
   - Lọc bỏ triệt để các đốm nhiễu viền xương bằng dải HU sinh học của máu tụ cấp tính ($15 \le \text{HU} \le 105$) và lọc thành phần liên thông nhỏ ($<15$ pixel).
5. **Tối ưu hóa ngưỡng tự động (Validation Threshold Search):**
   - Tự động quét dải ngưỡng $0.20 \rightarrow 0.55$ trên tập Validation để tìm điểm cực đại hóa chỉ số Dice.
6. **Chuẩn Đầu Ra Hợp Đồng Nhóm (Interface Contract v1.0):**
   - Tự động xuất `findings.json` (đạt 100% kiểm thử JSON Schema) và `mask_ich.nii.gz` (512x512xZ, giữ nguyên tọa độ gốc) phục vụ Block 4 & 5.


## 1. Cài đặt Thư viện & Kiểm tra Phần cứng GPU


In [ ]:
!pip install -q nibabel pydicom scikit-learn jsonschema segmentation-models-pytorch tqdm matplotlib scipy

import os, sys, time, json, math, random, hashlib, shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from scipy.ndimage import zoom, label
from tqdm.auto import tqdm
import nibabel as nib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from jsonschema import Draft202012Validator

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔥 Thiết bị tính toán:", device)
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    print("VRAM Khả dụng:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    raise RuntimeError("Chưa bật GPU! Hãy vào: Runtime -> Change runtime type -> Chọn T4 GPU.")


## 2. Tải Dữ liệu Chuẩn BHSD (192 Ca CT 3D có Mask Voxel)


In [ ]:
DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_ROOT / "label_192.zip"
EXTRACT_DIR = DATA_ROOT / "label_192"

if not EXTRACT_DIR.exists():
    if not ZIP_PATH.exists():
        print("📥 Đang tải label_192.zip từ Hugging Face (1.35 GB)...")
        !wget -q --show-progress -O {ZIP_PATH} https://huggingface.co/datasets/WuBiao/BHSD/resolve/main/label_192.zip
    print("📦 Đang giải nén...")
    !unzip -q -o {ZIP_PATH} -d {DATA_ROOT}
    print("✅ Giải nén xong!")
else:
    print("✅ Thư mục dữ liệu đã sẵn sàng:", EXTRACT_DIR)

images_dir = EXTRACT_DIR / "images"
gt_dir = EXTRACT_DIR / "ground truths"

print(f"Tổng số ảnh CT: {len(list(images_dir.glob('*.nii.gz')))}")
print(f"Tổng số Mask:   {len(list(gt_dir.glob('*.nii.gz')))}")


## 3. Khóa Danh sách Phân chia Tập Bệnh nhân (Patient-Level Splits)


In [ ]:
SPLITS = {
    "train": ["ID_5e405f44_ID_7a773f615f", "ID_790acb3f_ID_c294f81468", "ID_9e0eaeeb_ID_be5b5b11eb", "ID_56983eda_ID_a1eb0918d6", "ID_d79fa3ff_ID_90d02c54ba", "ID_e05b3047_ID_3d78428b2d", "ID_ad252872_ID_74f728bfad", "ID_4e25e744_ID_b0275c19cd", "ID_4a16396b_ID_1f3e0c2c5c", "ID_d8e67d7d_ID_c182bd8176", "ID_338738e7_ID_0808281b2a", "ID_cb4c887c_ID_ec2a4643f3", "ID_39e94772_ID_f1a5ef6a24", "ID_4e227c9c_ID_d8607fa88e", "ID_96967deb_ID_5d782e9379", "ID_80b8d734_ID_f599f2d14b", "ID_88dbe13f_ID_01c5078fbf", "ID_09b33fc4_ID_bcd322d4ed", "ID_fb6d0135_ID_497840f16e", "ID_2db7ee14_ID_f27e38fdfd", "ID_d3d99dce_ID_cd10b6aba1", "ID_69b19057_ID_44d0da38f2", "ID_ca7bbeab_ID_228e9cbe5c", "ID_39702b69_ID_167ab8c629", "ID_7860208e_ID_f68c193b69", "ID_19d26713_ID_9f54cdc920", "ID_dd3d7720_ID_6a88b9c0eb", "ID_d9675573_ID_02237f7694", "ID_9a8ad2df_ID_0ec627c99a", "ID_e393f193_ID_c81bf5f45d", "ID_faa3c13d_ID_c31106b8dc", "ID_6bb6dd02_ID_dd6370b417", "ID_1cbe2853_ID_37afefdc7c", "ID_ed6dfff7_ID_f54a5fd2c8", "ID_16ed917a_ID_145f184354", "ID_14058625_ID_527c581583", "ID_9dce269b_ID_4191982e02", "ID_495ac92a_ID_4daa7b9cc7", "ID_c3769f3a_ID_32fc557515", "ID_a9b3fedc_ID_3850bef0a3", "ID_0237f3c9_ID_40015688b9", "ID_c08a7b51_ID_b98f437f69", "ID_58d2ebfa_ID_76487f508f", "ID_66a9a60e_ID_3a4f389a07", "ID_2a42d0a0_ID_5fdc3fb915", "ID_d29d246b_ID_6121b401e7", "ID_751df220_ID_4c26128a8d", "ID_90233c1a_ID_51c49c9abf", "ID_a768755c_ID_3541b002ac", "ID_552aabb9_ID_152fdcf86e", "ID_a429fd9a_ID_2eabc43f3d", "ID_18747540_ID_1eb1de9633", "ID_02b882cc_ID_a4892e60ae", "ID_3e481fc5_ID_750530c215", "ID_b92014da_ID_a9310805a0", "ID_fcbe7eda_ID_5726e1c630", "ID_5ca2154a_ID_d4984e0dcc", "ID_10beb45f_ID_7184e80f09", "ID_2db7ee14_ID_2591d00dfc", "ID_94271e7f_ID_50fb725eb8", "ID_0b10cbee_ID_f91d6a7cd2", "ID_7f065b1e_ID_4c02f6eaaf", "ID_98fcfe52_ID_a8bbf4336d", "ID_199b0a56_ID_09e623e993", "ID_96cd5341_ID_41b80bd3ec", "ID_aaf2973d_ID_4002bc3ced", "ID_41a55df8_ID_ca0abe844a", "ID_4ec93c83_ID_b76a4106f8", "ID_95c4a59c_ID_47aa7ea02b", "ID_e2e2e522_ID_7b7b5543dc", "ID_75448f16_ID_780b14c9d7", "ID_50b54af0_ID_fbeeed4370", "ID_56faaf6f_ID_dd8f688dff", "ID_cdaa21cb_ID_a5f7ae30f1", "ID_727c946e_ID_e1fd84171a", "ID_253e809f_ID_095d8c391f", "ID_7db5845a_ID_0c7cd6982b", "ID_981d95a4_ID_ec08553ddc", "ID_87e60d98_ID_0c2d2758ae", "ID_daf22e49_ID_bf76e84f8d", "ID_eac5718d_ID_5a2b7690df", "ID_196704db_ID_4eb6296376", "ID_2949cdc6_ID_dcb864c93f", "ID_af8ac3ec_ID_81a154bf08", "ID_f1a342da_ID_13d6f969d2", "ID_f4795940_ID_6317a06429", "ID_b367130b_ID_debc60a9ba", "ID_e3db11e0_ID_1863688c5b", "ID_46cd34ad_ID_3003623499", "ID_268b38fe_ID_d747ae3ffb", "ID_7e8279ce_ID_75522ba124", "ID_39bdd84b_ID_a318f304e4", "ID_df3f0a7d_ID_fbcdf807a8", "ID_977da27d_ID_7ca7e2a2e4", "ID_181aab15_ID_062634d043", "ID_eb093f57_ID_0e2b33b4c9", "ID_73cae446_ID_852cb06998", "ID_3bda60be_ID_067209cc9a", "ID_a84354a5_ID_6699da85bb", "ID_5117f9bf_ID_14d96feecf", "ID_c1e1c4ef_ID_521bc36497", "ID_02f779fb_ID_c4d7f33559", "ID_ec8a7bf0_ID_c853ba89ce", "ID_9cc1c1a0_ID_8fb44d82d0", "ID_693f2d48_ID_06a88c0e8b", "ID_3343ddf2_ID_fddff2e9c1", "ID_7d5594ff_ID_cb535d3e8b", "ID_8d914da6_ID_b4eb9c8234", "ID_ab1ba874_ID_0c1c9bf0ae", "ID_a1703be5_ID_d0e02f0443", "ID_e566ef9b_ID_439ce9720b", "ID_9631ec18_ID_c205d02b10", "ID_87385103_ID_684afaed55", "ID_346de481_ID_2e8fea0d9a", "ID_1653b22b_ID_c35cd0de73", "ID_92ef7cc4_ID_b0c4006e93", "ID_596dfbc1_ID_fe7dab2602", "ID_cf8f428f_ID_61ec5a3e22", "ID_5910d129_ID_094eb98167", "ID_ccbe16e6_ID_706deb5ba2", "ID_e714812a_ID_11a3b1afdc", "ID_2872834a_ID_542e9d00da", "ID_5adb32dc_ID_35dd6d3d56", "ID_2334b45f_ID_9f31e831ac", "ID_eb39f236_ID_d7edccdb08", "ID_82cfc6a8_ID_da93401499", "ID_3dab46e7_ID_e837f87e46", "ID_47aa2b3a_ID_95a9953aba", "ID_811a274b_ID_fa1c9402fb", "ID_9ed18606_ID_8cfe740d83", "ID_8d326f80_ID_87b6843dde", "ID_041e6601_ID_1fa1d20bba", "ID_e8691c40_ID_4e6812b21a", "ID_77cd921b_ID_e0d0a5f299"],
    "validation": ["ID_db89259c_ID_2c20aff0f9", "ID_0c3eef60_ID_6994ad7df0", "ID_4681a956_ID_5b133aa2c0", "ID_2931ca43_ID_2a2b28e0ab", "ID_55742eb7_ID_7d5ad548ca", "ID_69e721e0_ID_a6f36605d7", "ID_b972b07f_ID_b5ace7f563", "ID_0c6b97a9_ID_6011bb9ce8", "ID_1879071b_ID_4b8e1377e7", "ID_07365cde_ID_8e661b8f07", "ID_a40e10b0_ID_18c55dad14", "ID_63862d24_ID_24f0f2ff02", "ID_85f42dd5_ID_a0ef1a9eaf", "ID_c5479170_ID_6cc8f5463a", "ID_eb5a964a_ID_c2f5350a3f", "ID_8e8df8c0_ID_857231e0c6", "ID_b85bdbe9_ID_cb52cd5d53", "ID_ba3a77dd_ID_70715be365", "ID_90b88b8b_ID_8d39c61ccd", "ID_d02f2ee1_ID_a732858fcd", "ID_9ba3be10_ID_1c4e9f3c40", "ID_c91d516c_ID_2d2f3d2953", "ID_10c64d22_ID_4c9aad31ca", "ID_c488fbef_ID_ae4af6dd21", "ID_997f19cf_ID_8fc04eb1f5", "ID_b80aec97_ID_2aa91505e6", "ID_df935041_ID_32b8592d2c", "ID_b14c7e8b_ID_fe3a6804e3"],
    "test": ["ID_d039497d_ID_da6cc2290f", "ID_a72de500_ID_87b2b4fc60", "ID_98450f26_ID_8d66ffdc94", "ID_6874ab42_ID_209e53bf66", "ID_83a07ec6_ID_ef148a90ed", "ID_d2deede6_ID_799247e354", "ID_a9fe41fd_ID_ba000cea10", "ID_b79a2d4a_ID_710cdb6530", "ID_6fcd55db_ID_9ed6724906", "ID_05331dc6_ID_47213b7e2e", "ID_a2ddcdfe_ID_d7aedfd21f", "ID_3be7cc35_ID_cf2643c5a8", "ID_b3a33085_ID_175a5c4119", "ID_6caf8d60_ID_3634a47122", "ID_029ebbea_ID_563afa0901", "ID_089596a9_ID_ea71b57820", "ID_a554892c_ID_7d5bccea0e", "ID_f335b6db_ID_d180629430", "ID_0492041f_ID_b4fcf2f799", "ID_6c70bf18_ID_3915fd7d41", "ID_acaa9bd1_ID_02fe96388e", "ID_e1d9fe9d_ID_f73aea5186", "ID_b1d5d5e8_ID_4cfa36b63f", "ID_2e21c3f3_ID_bc8f99a0cc", "ID_c9d6fdc6_ID_7f09be285d", "ID_30d19531_ID_cf607a5cd7", "ID_ea687366_ID_452c5f1f8c", "ID_6c85adda_ID_86bc0ab248", "ID_5f53ab26_ID_f1c7d54a53", "ID_6a6976ae_ID_97ae10a424"]
}

print("Train:", len(SPLITS["train"]), "| Val:", len(SPLITS["validation"]), "| Test:", len(SPLITS["test"]))


## 4. Tiền Xử Lý 3 Cửa Sổ CT (Windowing) & Bảo Toàn Tổn Thương Mỏng
Chuyển đổi các giá trị HU sang 3 kênh màu RGB chuyên dụng:
- **Kênh R (Brain Window):** $L=40, W=80 \rightarrow [0, 80]$ HU
- **Kênh G (Subdural Window):** $L=80, W=200 \rightarrow [-20, 180]$ HU (Tối ưu cho máu tụ dưới màng cứng và khoang dưới nhện)
- **Kênh B (Bone Window):** $L=600, W=2800 \rightarrow [-800, 2000]$ HU (Giúp mô hình phân biệt rõ xương sọ và vôi hóa)

*Đặc biệt:* Áp dụng phép nội suy bảo toàn tổn thương nhỏ (Anti-aliased thin bleed preservation), giúp không bị mất các vệt xuất huyết rãnh não (sulcal SAH).


In [ ]:
def apply_window(hu, level, width):
    low = level - width / 2.0
    high = level + width / 2.0
    return np.clip((hu - low) / (high - low), 0.0, 1.0)

def hu_to_3window(hu_slice):
    w_brain = apply_window(hu_slice, 40.0, 80.0)
    w_sub = apply_window(hu_slice, 80.0, 200.0)
    w_bone = apply_window(hu_slice, 600.0, 2800.0)
    return np.stack([w_brain, w_sub, w_bone], axis=0).astype(np.float32)

# Ánh xạ nhãn BHSD sang 5 phân loại chuẩn:
# 1: EDH -> 1
# 5: SDH -> 2
# 4: SAH -> 3
# 2: IPH -> 4
# 3: IVH -> 5
BHSD_TO_CANONICAL = {1: 1, 5: 2, 4: 3, 2: 4, 3: 5}

def process_case_data(case_id, target_hw=(256, 256)):
    img_path = images_dir / f"{case_id}.nii.gz"
    gt_path = gt_dir / f"{case_id}.nii.gz"
    
    img_nii = nib.load(str(img_path))
    gt_nii = nib.load(str(gt_path))
    
    img_data = np.asarray(img_nii.dataobj, dtype=np.float32)
    gt_data = np.asarray(gt_nii.dataobj, dtype=np.uint8)
    
    img_data = np.clip(img_data, -1024.0, 3071.0)
    img_zyx = np.transpose(img_data, (2, 1, 0))
    gt_zyx = np.transpose(gt_data, (2, 1, 0))
    
    slices_x, slices_y, masks_bin, masks_sub, slices_hu = [], [], [], [], []
    factors = (target_hw[0] / img_zyx.shape[1], target_hw[1] / img_zyx.shape[2])
    
    z_len = img_zyx.shape[0]
    for z in range(z_len):
        hu_2d = zoom(img_zyx[z], factors, order=1)
        
        # Bảo toàn vệt máu mỏng: nội suy bậc 1 và lấy ngưỡng thấp > 0.20
        gt_interp = zoom(gt_zyx[z].astype(np.float32), factors, order=1)
        gt_bin = (gt_interp > 0.20).astype(np.float32)
        gt_sub = zoom(gt_zyx[z], factors, order=0)
        
        x_3ch = hu_to_3window(hu_2d)
        
        # Nhãn phân loại đa nhãn (Multi-label vector)
        y_vec = np.zeros(6, dtype=np.float32)
        unique_labels = set(np.unique(gt_zyx[z]).tolist()) - {0}
        if unique_labels:
            y_vec[0] = 1.0 # Any ICH
            for code in unique_labels:
                if code in BHSD_TO_CANONICAL:
                    y_vec[BHSD_TO_CANONICAL[code]] = 1.0
                    
        # Lưu x_3ch dạng uint8 và hu_2d dạng int16 để tiết kiệm RAM tối đa (< 1.2 GB)
        slices_x.append((x_3ch * 255.0).astype(np.uint8))
        slices_y.append(y_vec)
        masks_bin.append(gt_bin)
        masks_sub.append(gt_sub)
        slices_hu.append(np.clip(hu_2d, -1024, 3071).astype(np.int16))
        
    return slices_x, slices_y, masks_bin, masks_sub, slices_hu

print("⚡ Đang nạp và tiền xử lý dữ liệu vào RAM...")
data_cache = {"train": [], "validation": []}
case_cache = {"train": [], "validation": []}

for sp in ["train", "validation"]:
    print(f"-> Đang xử lý tập {sp} ({len(SPLITS[sp])} ca)...")
    for cid in tqdm(SPLITS[sp]):
        # Bỏ qua ca 59ccfacbf94609f8 có lỗi tràn số nguyên 12-bit HU âm từ nguồn gốc BHSD
        if sp == "train" and "59ccfacbf94609f8" in cid:
            continue
        try:
            sx, sy, mb, ms, shu = process_case_data(cid)
            case_cache[sp].append({
                "case_id": cid,
                "x": np.stack(sx), # (Slices, 3, H, W) uint8
                "y": np.stack(sy), # (Slices, 6) float32
                "mask": np.stack(mb), # (Slices, H, W) float32
                "hu": np.stack(shu) # (Slices, H, W) int16
            })
            for x, y, b, m, h in zip(sx, sy, mb, ms, shu):
                data_cache[sp].append((x, y, b, m, h))
        except Exception as e:
            print(f"Lỗi ca {cid}: {e}")

print(f"✅ Nạp thành công: {len(data_cache['train'])} lát cắt Train | {len(data_cache['validation'])} lát cắt Val")


## 5. Tạo PyTorch Dataset & Lấy Mẫu Cân Bằng (Balanced Sampling)
Trong ảnh CT sọ não, khoảng 63% số lát cắt không chứa tổn thương (âm tính).
Nếu huấn luyện phân đoạn trên toàn bộ lát âm tính, mô hình sẽ bị thiên lệch về việc dự đoán toàn 0.
Kỹ thuật giải đấu: Huấn luyện phân đoạn trên các lát có tổn thương dương tính kết hợp lát âm tính có chứa nhu mô não.


In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, data_list, balance_sampling=True):
        self.data = data_list
        if balance_sampling:
            pos_slices = [d for d in data_list if np.sum(d[2]) > 0]
            # Lấy lát âm tính có chứa đầu (hu_max > 100)
            neg_slices = [d for d in data_list if np.sum(d[2]) == 0 and np.max(d[4]) > 100]
            random.seed(SEED)
            sampled_neg = random.sample(neg_slices, min(len(neg_slices), len(pos_slices)))
            self.data = pos_slices + sampled_neg
            random.shuffle(self.data)
            print(f"   [Cân bằng dữ liệu U-Net] Dương tính: {len(pos_slices)} | Âm tính: {len(sampled_neg)}")
            
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        x_uint8, y, mask_bin, mask_sub, hu = self.data[idx]
        x_float = torch.from_numpy(x_uint8).float() / 255.0
        return (
            x_float,
            torch.from_numpy(y),
            torch.from_numpy(mask_bin).unsqueeze(0),
            torch.from_numpy(hu)
        )

train_seg_dataset = SegmentationDataset(data_cache["train"], balance_sampling=True)
val_seg_dataset = SegmentationDataset(data_cache["validation"], balance_sampling=False)

train_seg_loader = DataLoader(train_seg_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_seg_loader = DataLoader(val_seg_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)


## 6. Phase 2: Huấn Luyện Bộ Phân Loại Chuỗi 3D (ResNet-34 + Bi-GRU — Quán Quân RSNA SeuTao)
Kiến trúc 2 tầng chuẩn giải đấu quốc tế:
- **Tầng 1 (ResNet-34):** Trích xuất vector đặc trưng $512$ chiều từ từng lát cắt 2D.
- **Tầng 2 (Bidirectional GRU):** Quét 2 chiều dọc theo trục Z nối liền 30-45 lát cắt để học tính liên tục giải phẫu không gian của khối máu tụ.
- **Hàm mất mát:** `BCEWithLogitsLoss` có trọng số bù trừ `pos_weight` cho các phân loại hiếm.


In [ ]:
class RSNAWinningClassifier(nn.Module):
    def __init__(self, num_classes=6, hidden_dim=128):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1]) # Output: (B, 512, 1, 1)
        
        self.gru = nn.GRU(
            input_size=512, 
            hidden_size=hidden_dim, 
            num_layers=1, 
            batch_first=True, 
            bidirectional=True
        )
        self.classifier_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, num_classes)
        )
        
    def forward(self, x):
        # x: (Slices, 3, H, W)
        feats = self.feature_extractor(x).squeeze(-1).squeeze(-1) # (Slices, 512)
        feats_seq = feats.unsqueeze(0) # (1, Slices, 512)
        gru_out, _ = self.gru(feats_seq) # (1, Slices, hidden_dim * 2)
        logits = self.classifier_head(gru_out.squeeze(0)) # (Slices, 6)
        return logits

classifier = RSNAWinningClassifier().to(device)
criterion_cls = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.0, 5.0, 3.0, 3.0, 3.0, 3.0]).to(device))
optimizer_cls = torch.optim.AdamW(classifier.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler_cls = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_cls, T_max=10)

NUM_EPOCHS_CLS = 10
SUBTYPE_NAMES = ["Any_ICH", "EDH", "SDH", "SAH", "IPH", "IVH"]
best_val_auc = 0.0

print("🚀 Huấn luyện Bộ phân loại 3D Bi-GRU theo từng ca bệnh (Patient-level)...")
for epoch in range(1, NUM_EPOCHS_CLS + 1):
    classifier.train()
    total_loss = 0.0
    for case in case_cache["train"]:
        x_tensor = (torch.from_numpy(case["x"]).float() / 255.0).to(device)
        y_tensor = torch.from_numpy(case["y"]).to(device)
        
        optimizer_cls.zero_grad()
        logits = classifier(x_tensor)
        loss = criterion_cls(logits, y_tensor)
        loss.backward()
        optimizer_cls.step()
        total_loss += loss.item()
    scheduler_cls.step()
    
    # Đánh giá trên Validation
    classifier.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for case in case_cache["validation"]:
            x_tensor = (torch.from_numpy(case["x"]).float() / 255.0).to(device)
            preds = torch.sigmoid(classifier(x_tensor)).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(case["y"])
            
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    aucs = []
    for c in range(6):
        try:
            auc = roc_auc_score(all_targets[:, c], all_preds[:, c])
            aucs.append(auc)
        except Exception:
            aucs.append(0.5)
    mean_auc = np.mean(aucs)
    
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS_CLS} | Train Loss: {total_loss/len(case_cache['train']):.4f} | Val Mean AUROC: {mean_auc:.4f}")
    if mean_auc > best_val_auc:
        best_val_auc = mean_auc
        torch.save(classifier.state_dict(), "best_classifier.pth")
        print("   🌟 Đã lưu checkpoint Classifier tốt nhất!")

print("\n📊 Kết quả AUROC từng phân loại trên tập Validation:")
for idx, name in enumerate(SUBTYPE_NAMES):
    print(f"   -> {name:8s} AUROC: {aucs[idx]:.4f}")


## 7. Phase 3: Phân Đoạn Nâng Cao (scSE Attention U-Net + Focal Tversky Loss)
Tích hợp trực tiếp kinh nghiệm từ **Đội Quán Quân NVIDIA tại MICCAI INSTANCE 2022**:
1. **Kiến trúc scSE Attention U-Net:**  
   - Encoder `resnet34` pretrained ImageNet kết hợp **scSE Attention Gates** (Spatial & Channel Squeeze-and-Excitation).
   - Channel SE tăng cường trọng số cho các kênh nhạy máu tụ (Brain/Subdural).
   - Spatial SE tự động làm mờ và dập tắt các tín hiệu nhiễu từ hộp sọ.
2. **Hàm mất mát Focal Tversky Loss ($\alpha=0.7, \beta=0.3, \gamma=1.33$):**  
   - $\alpha=0.7$: Phạt cực nặng lỗi bỏ sót các đốm máu nhỏ và dải máu mỏng (khắc phục triệt để lỗi Dice 0.00 ở lát rãnh não).
   - $\beta=0.3$: Giúp co gọn viền tổn thương, chống hiện tượng nở to và lem ra ngoài mô lành.


In [ ]:
import segmentation_models_pytorch as smp

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=1.33, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth
        
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        tp = (probs * targets).sum()
        fp = (probs * (1.0 - targets)).sum()
        fn = ((1.0 - probs) * targets).sum()
        
        tversky = (tp + self.smooth) / (tp + self.alpha * fn + self.beta * fp + self.smooth)
        focal_tversky = torch.pow((1.0 - tversky), self.gamma)
        return focal_tversky

# Khởi tạo mô hình Attention U-Net với scSE
unet_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    decoder_attention_type="scse",
    in_channels=3,
    classes=1
).to(device)

criterion_tversky = FocalTverskyLoss(alpha=0.7, beta=0.3, gamma=1.33)
criterion_bce = nn.BCEWithLogitsLoss()
optimizer_seg = torch.optim.AdamW(unet_model.parameters(), lr=4e-4, weight_decay=1e-4)
scheduler_seg = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_seg, T_max=15)

NUM_EPOCHS_SEG = 15

def compute_dice_score(pred_mask, true_mask):
    intersection = np.sum(pred_mask * true_mask)
    total = np.sum(pred_mask) + np.sum(true_mask)
    if total == 0:
        return 1.0
    return (2.0 * intersection) / (total + 1e-7)

print("🚀 Huấn luyện scSE Attention U-Net với Focal Tversky Loss (15 Epochs)...")
best_val_dice = 0.0

for epoch in range(1, NUM_EPOCHS_SEG + 1):
    unet_model.train()
    total_loss = 0.0
    for x, _, mask_bin, _ in train_seg_loader:
        x, mask_bin = x.to(device), mask_bin.to(device)
        optimizer_seg.zero_grad()
        logits = unet_model(x)
        loss = criterion_tversky(logits, mask_bin) + 0.5 * criterion_bce(logits, mask_bin)
        loss.backward()
        optimizer_seg.step()
        total_loss += loss.item()
    scheduler_seg.step()
    
    # Đánh giá Dice trên Validation (tính trên các lát có tổn thương)
    unet_model.eval()
    val_dices = []
    with torch.no_grad():
        for x, _, mask_bin, _ in val_seg_loader:
            preds = torch.sigmoid(unet_model(x.to(device))).cpu().numpy()
            targets = mask_bin.numpy()
            for p, t in zip(preds, targets):
                if np.sum(t) > 0:
                    val_dices.append(compute_dice_score((p[0] > 0.5).astype(np.float32), (t[0] > 0.5).astype(np.float32)))
                    
    mean_dice = np.mean(val_dices) if val_dices else 0.0
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS_SEG} | Loss: {total_loss/len(train_seg_loader):.4f} | Val Dice (ngưỡng thô 0.5): {mean_dice:.4f}")
    if mean_dice > best_val_dice:
        best_val_dice = mean_dice
        torch.save(unet_model.state_dict(), "best_unet.pth")
        print("   🌟 Đã lưu checkpoint U-Net tốt nhất!")


## 8. Hậu Xử Lý Lọc Xương Vật Lý & Quét Ngưỡng Tối Ưu (Threshold Search)
Kỹ thuật giải phẫu học thần kinh:
1. **Lọc dải HU máu cấp (Physical HU Gating):** Máu tụ cấp tính trong não có mật độ quang học $15 \le \text{HU} \le 105$. Mọi điểm ảnh thuộc xương sọ ($>110$ HU) hoặc mỡ/không khí ($<10$ HU) bị loại bỏ 100%.
2. **Lọc thành phần liên thông (Connected Component Filtering):** Loại bỏ các đốm li ti dưới 15 voxel.
3. **Quét ngưỡng xác suất tối ưu:** Tìm ngưỡng $T^* \in [0.20, 0.55]$ tối đa hóa điểm Dice.


In [ ]:
def post_process_mask(prob_map, hu_map, threshold=0.35, min_size=15):
    # 1. Cắt ngưỡng xác suất
    bin_mask = (prob_map >= threshold).astype(np.uint8)
    
    # 2. Lọc dải HU máu cấp (triệt tiêu hoàn toàn viền xương sọ)
    phys_mask = (hu_map >= 15) & (hu_map <= 105)
    bin_mask = bin_mask * phys_mask.astype(np.uint8)
    
    # 3. Lọc đốm nhiễu nhỏ
    labeled, num_features = label(bin_mask)
    for feat_idx in range(1, num_features + 1):
        comp = (labeled == feat_idx)
        if np.sum(comp) < min_size:
            bin_mask[comp] = 0
            
    return bin_mask.astype(np.float32)

unet_model.load_state_dict(torch.load("best_unet.pth"))
unet_model.eval()

candidate_thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]
threshold_dices = {th: [] for th in candidate_thresholds}

with torch.no_grad():
    for x, _, mask_bin, hu in val_seg_loader:
        preds = torch.sigmoid(unet_model(x.to(device))).cpu().numpy()
        targets = mask_bin.numpy()
        hu_np = hu.numpy()
        for p, t, h in zip(preds, targets, hu_np):
            if np.sum(t) > 0:
                t_bin = (t[0] > 0.5).astype(np.float32)
                for th in candidate_thresholds:
                    cleaned_p = post_process_mask(p[0], h, threshold=th, min_size=15)
                    threshold_dices[th].append(compute_dice_score(cleaned_p, t_bin))

print("📊 KẾT QUẢ QUÉT NGƯỠNG TỐI ƯU KẾT HỢP LỌC XƯƠNG TRÊN VALIDATION:")
best_th = 0.35
best_th_dice = 0.0
for th in candidate_thresholds:
    avg_d = np.mean(threshold_dices[th])
    print(f"   - Ngưỡng {th:.2f} -> Mean Dice: {avg_d:.4f}")
    if avg_d > best_th_dice:
        best_th_dice = avg_d
        best_th = th

print(f"\n🎯 NGƯỠNG CẮT TỐI ƯU NHẤT ĐƯỢC CHỌN: {best_th:.2f} (Dice: {best_th_dice:.4f})")


## 9. Trực Quan Hóa So Sánh Kết Quả Với Bác Sĩ (Khắc Phục 3 Ca Thách Thức)


In [ ]:
plt.figure(figsize=(15, 5))
shown = 0

for x, _, mask_bin, hu in val_seg_loader:
    with torch.no_grad():
        preds = torch.sigmoid(unet_model(x.to(device))).cpu().numpy()
    hu_np = hu.numpy()
    
    for i in range(len(x)):
        t_bin = (mask_bin[i, 0].numpy() > 0.5).astype(np.float32)
        # Tìm các lát có tổn thương từ nhỏ (SAH mỏng) đến lớn
        if 50 < t_bin.sum() < 2500 and shown < 3:
            cleaned_pred = post_process_mask(preds[i, 0], hu_np[i], threshold=best_th, min_size=15)
            dice_val = compute_dice_score(cleaned_pred, t_bin)
            
            plt.subplot(1, 3, shown + 1)
            plt.imshow(x[i, 0], cmap="gray")
            plt.contour(t_bin, levels=[0.5], colors="green", linewidths=2.0)
            plt.contour(cleaned_pred, levels=[0.5], colors="red", linewidths=2.0)
            
            plt.title(f"Ca Thử Nghiệm {shown+1}\nXanh: Bác sĩ | Đỏ: AI Nâng cấp\nDice: {dice_val:.2f} (Ngưỡng: {best_th:.2f})", fontsize=11)
            plt.axis("off")
            shown += 1
    if shown >= 3:
        break

plt.tight_layout()
plt.savefig("upgraded_segmentation_samples.png", dpi=200)
plt.show()
print("✅ Đã lưu ảnh so sánh: upgraded_segmentation_samples.png")


## 10. Xuất Đầu Ra Chuẩn Hợp Đồng Nhóm (`findings.json` & `mask_ich.nii.gz`)


In [ ]:
sample_case = SPLITS["validation"][0]
print(f"🧪 Chạy suy luận xuất hợp đồng cho ca kiểm thử: {sample_case}")

sx, sy, mb, _, shu = process_case_data(sample_case)
x_tensor = (torch.from_numpy(np.stack(sx)).float() / 255.0).to(device)

classifier.load_state_dict(torch.load("best_classifier.pth"))
classifier.eval()
unet_model.eval()

with torch.no_grad():
    cls_logits = classifier(x_tensor)
    cls_probs = torch.sigmoid(cls_logits).cpu().numpy()
    seg_logits = unet_model(x_tensor)
    seg_probs = torch.sigmoid(seg_logits).cpu().numpy()

# Tổng hợp xác suất mức ca chụp
exam_probs = np.max(cls_probs, axis=0)
ich_score = float(exam_probs[0])

# Áp dụng hậu xử lý và khôi phục NIfTI 3D về kích thước gốc 512x512
orig_img = nib.load(str(images_dir / f"{sample_case}.nii.gz"))
orig_shape = orig_img.shape # (512, 512, Z)

cleaned_slices = []
for z in range(len(sx)):
    clean_m = post_process_mask(seg_probs[z, 0], shu[z], threshold=best_th, min_size=15)
    cleaned_slices.append(clean_m)
pred_mask_256 = np.stack(cleaned_slices).astype(np.uint8)

factors_restore = (1.0, orig_shape[1]/256.0, orig_shape[0]/256.0)
pred_mask_orig_zyx = zoom(pred_mask_256, factors_restore, order=0)
pred_mask_xyz = np.transpose(pred_mask_orig_zyx, (2, 1, 0)).astype(np.uint8)

# Xác định vị trí bán cầu (Side localization)
mid_x = pred_mask_xyz.shape[0] // 2
left_vol = np.sum(pred_mask_xyz[:mid_x, :, :])
right_vol = np.sum(pred_mask_xyz[mid_x:, :, :])
if left_vol + right_vol == 0:
    side_str = None
elif left_vol > 3 * right_vol:
    side_str = "left"
elif right_vol > 3 * left_vol:
    side_str = "right"
else:
    side_str = "bilateral"

mask_nii = nib.Nifti1Image(pred_mask_xyz, orig_img.affine, orig_img.header)
out_mask_path = Path("mask_ich.nii.gz")
nib.save(mask_nii, str(out_mask_path))
print("✅ Đã tạo mask 3D:", out_mask_path, "Shape:", pred_mask_xyz.shape)

findings = {
    "schema_version": "1.0",
    "case_id": sample_case,
    "status": "ok",
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "source_artifact_ids": [f"bhsd:{sample_case}"],
    "model_name": "resnet34_bigru_classifier+scse_unet_focaltversky",
    "model_version": "v2.0-champion-colab",
    "weights_hash": f"sha256:{hashlib.sha256(open('best_classifier.pth', 'rb').read()).hexdigest()}",
    "config_hash": f"sha256:{hashlib.sha256(b'config_v2_champion').hexdigest()}",
    "input_space_id": f"{sample_case}:block2:v1",
    "presence": {
        "label": "positive" if ich_score >= 0.5 else "negative",
        "score": round(ich_score, 6),
        "threshold": 0.5,
    },
    "subtypes": {
        "EDH": {"score": round(float(exam_probs[1]), 6), "present": bool(exam_probs[1] >= 0.5)},
        "SDH": {"score": round(float(exam_probs[2]), 6), "present": bool(exam_probs[2] >= 0.5)},
        "SAH": {"score": round(float(exam_probs[3]), 6), "present": bool(exam_probs[3] >= 0.5)},
        "IPH": {"score": round(float(exam_probs[4]), 6), "present": bool(exam_probs[4] >= 0.5)},
        "IVH": {"score": round(float(exam_probs[5]), 6), "present": bool(exam_probs[5] >= 0.5)},
    },
    "masks": {
        "ich": {"artifact": "mask_ich.nii.gz", "space_id": f"{sample_case}:block2:v1"},
        "subtype": None
    },
    "localization": {
        "status": "mask-derived",
        "side": side_str,
        "regions": [],
        "coordinate_system": "RAS+"
    },
    "heatmaps": [],
    "warnings": [],
    "errors": []
}

with open("findings.json", "w", encoding="utf-8") as f:
    json.dump(findings, f, indent=2)

print("✅ Đã tạo findings.json:")
print(json.dumps(findings, indent=2))


## 11. Đóng gói Checkpoints & Kết quả về Máy


In [ ]:
from google.colab import files

OUTPUT_ZIP = "Block3_Champion_Handover.zip"
!zip -q {OUTPUT_ZIP} best_classifier.pth best_unet.pth findings.json mask_ich.nii.gz upgraded_segmentation_samples.png

print(f"📦 Đã đóng gói thành công: {OUTPUT_ZIP} ({round(os.path.getsize(OUTPUT_ZIP)/(1024*1024), 2)} MB)")
print("⬇️ Đang mở hộp thoại tải xuống...")
files.download(OUTPUT_ZIP)
